In [1]:
# ============ threadpoolctl monkey-patch（必须最前）============
import threadpoolctl
class _SafeThreadpoolLimits:
    def __init__(self, *args, **kwargs): pass
    def __enter__(self): return self
    def __exit__(self, *args): pass
threadpoolctl.threadpool_limits = _SafeThreadpoolLimits
# ================================================================

import os, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              precision_recall_curve, f1_score,
                              recall_score, precision_score,
                              confusion_matrix, brier_score_loss)
from sklearn.isotonic import IsotonicRegression
from imblearn.over_sampling import SMOTE
from scipy import stats
import lightgbm as lgb

data_path = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\建模数据集_方案A_MDA_未隔离.csv"
out_dir   = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"

# ================= 加载数据 =================
data = pd.read_csv(data_path)
exclude_cols = ["Stkcd", "year", "Fraud", "ShortName", "IndustryName1",
                "ViolationTypeID", "DeclareDate", "DisposalDate", "Enddate", "set"]
all_numeric = [c for c in data.columns
               if c not in exclude_cols and pd.api.types.is_numeric_dtype(data[c])]

y      = data["Fraud"].values
groups = data["Stkcd"].values
X_all  = data[all_numeric].values

MDA_FEATURES = [
    "TextualSimilarity", "PositiveVocabularyNum", "NegativeVocabularyNum",
    "EmotionTone1", "EmotionTone2",
    "PosRatio", "NegRatio", "SentLenAvg", "SentLenStd", "ComplexWordRatio",
    "DigitDensity", "PuncDensity", "TTR",
    "Jaccard_prev", "EditSim_prev", "TFIDF_Cosine_prev",
    "DLUT_PosNum", "DLUT_NegNum", "DLUT_PosRatio", "DLUT_NegRatio",
    "DLUT_PosIntensity", "DLUT_NegIntensity", "DLUT_EmotionScore",
    "DLUT_EmotionTone", "DLUT_NegAfterNeg", "DLUT_PosAfterNeg",
]
MDA_FEATURES = [f for f in MDA_FEATURES if f in all_numeric]

print(f"特征数: {len(all_numeric)}   样本数: {len(y)}   正例: {int(y.sum())}")
print(f"MD&A 特征数: {len(MDA_FEATURES)}")

# ================= 固定参数（沿用之前 Optuna 的中庸参数）=================
# 注意：不加 class_weight='balanced'，因为 SMOTE 已经平衡了训练集
best_params = dict(
    n_estimators=400, learning_rate=0.015, num_leaves=25,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.75,
    reg_alpha=0.15, reg_lambda=0.2,
    random_state=42, n_jobs=-1, verbose=-1
)

def find_best_threshold(y_true, y_prob):
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    return float(thr[np.argmax(f1s)])

def compute_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return dict(
        AUC=roc_auc_score(y_true, y_prob),
        PR_AUC=average_precision_score(y_true, y_prob),
        F1=f1_score(y_true, y_pred, zero_division=0),
        Recall=recall_score(y_true, y_pred, zero_division=0),
        Precision=precision_score(y_true, y_pred, zero_division=0),
        Specificity=tn / (tn + fp) if (tn + fp) > 0 else np.nan,
        Brier=brier_score_loss(y_true, y_prob),
    )

# ================= SMOTE 5 折主实验 =================
SMOTE_RATIO = 'auto'   # 'auto' = 1:1；可以改成 0.5 (1:2) 做保守版
SMOTE_K = 5

gkf = GroupKFold(n_splits=5)
probs_raw = np.zeros(len(y))   # SMOTE 训练后的原始概率
probs_cal = np.zeros(len(y))   # 校准后
shap_all = np.zeros((len(y), len(all_numeric)))
fold_ids = np.zeros(len(y), dtype=int)
metrics_raw, metrics_cal = [], []
fold_info = []

t0 = time.time()
for fold, (tr, te) in enumerate(gkf.split(X_all, y, groups), 1):
    print(f"\n===== Fold {fold} =====")
    X_tr_raw, X_te_raw = X_all[tr], X_all[te]
    y_tr, y_te = y[tr], y[te]
    g_tr = groups[tr]

    # ---- 1. 先 impute（SMOTE 不能处理 NaN）----
    imp = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(X_tr_raw)
    X_te = imp.transform(X_te_raw)

    # ---- 2. SMOTE（只在训练集）----
    smote = SMOTE(sampling_strategy=SMOTE_RATIO, k_neighbors=SMOTE_K,
                  random_state=42, n_jobs=-1)
    X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)
    print(f"  SMOTE: {X_tr.shape[0]} -> {X_tr_res.shape[0]}  "
          f"(pos: {y_tr.sum()} -> {y_tr_res.sum()})")

    # ---- 3. 训练模型 ----
    m = lgb.LGBMClassifier(**best_params)
    m.fit(X_tr_res, y_tr_res)
    p_raw = m.predict_proba(X_te)[:, 1]
    probs_raw[te] = p_raw
    fold_ids[te] = fold

    # ---- 4. 内层 3 折 OOF，用于校准 ----
    inner_gkf = GroupKFold(n_splits=3)
    oof_prob = np.zeros(len(tr))
    for i_tr, i_val in inner_gkf.split(X_tr, y_tr, g_tr):
        imp_i = SimpleImputer(strategy='median')
        X_i = imp_i.fit_transform(X_tr[i_tr])
        X_v = imp_i.transform(X_tr[i_val])
        sm_i = SMOTE(sampling_strategy=SMOTE_RATIO, k_neighbors=SMOTE_K,
                     random_state=42, n_jobs=-1)
        X_i_res, y_i_res = sm_i.fit_resample(X_i, y_tr[i_tr])
        mi = lgb.LGBMClassifier(**best_params)
        mi.fit(X_i_res, y_i_res)
        oof_prob[i_val] = mi.predict_proba(X_v)[:, 1]

    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(oof_prob, y_tr)
    p_cal = iso.predict(p_raw)
    probs_cal[te] = p_cal

    # ---- 5. 阈值：内层 OOF 校准后 F1 最优 ----
    oof_cal = iso.predict(oof_prob)
    thr = find_best_threshold(y_tr, oof_cal)

    # ---- 6. 指标 ----
    met_raw = compute_metrics(y_te, p_raw, 0.5)
    met_cal = compute_metrics(y_te, p_cal, thr)
    met_raw['fold'] = fold; met_cal['fold'] = fold
    met_raw['thr'] = 0.5; met_cal['thr'] = thr
    metrics_raw.append(met_raw); metrics_cal.append(met_cal)
    fold_info.append(dict(fold=fold, thr=thr,
                          train_orig=len(tr), train_smote=len(X_tr_res),
                          test=len(te), test_pos=int(y_te.sum())))

    print(f"  thr={thr:.3f}")
    print(f"  raw  AUC={met_raw['AUC']:.4f} PR={met_raw['PR_AUC']:.4f} Brier={met_raw['Brier']:.4f}")
    print(f"  cal  AUC={met_cal['AUC']:.4f} PR={met_cal['PR_AUC']:.4f} "
          f"F1={met_cal['F1']:.4f} R={met_cal['Recall']:.4f} "
          f"P={met_cal['Precision']:.4f} Spec={met_cal['Specificity']:.4f} "
          f"Brier={met_cal['Brier']:.4f}")
    print(f"  耗时: {(time.time()-t0)/60:.2f} min")

print(f"\n总耗时: {(time.time()-t0)/60:.2f} 分钟")

# ================= 5 折汇总 =================
df_raw = pd.DataFrame(metrics_raw)
df_cal = pd.DataFrame(metrics_cal)
df_fold = pd.DataFrame(fold_info)

print("\n" + "=" * 80)
print("SMOTE 5 折均值")
print("=" * 80)
for col in ['AUC','PR_AUC','F1','Recall','Precision','Specificity','Brier']:
    print(f"  {col:12s}: raw {df_raw[col].mean():.4f}±{df_raw[col].std():.4f}  "
          f"cal {df_cal[col].mean():.4f}±{df_cal[col].std():.4f}")

df_raw.to_csv(os.path.join(out_dir, "SMOTE_calibration_raw.csv"),
              index=False, encoding='utf-8-sig')
df_cal.to_csv(os.path.join(out_dir, "SMOTE_calibration_calibrated.csv"),
              index=False, encoding='utf-8-sig')
df_fold.to_csv(os.path.join(out_dir, "SMOTE_fold_info.csv"),
               index=False, encoding='utf-8-sig')
pd.DataFrame({
    'y_true': y,
    'y_prob_raw': probs_raw,
    'y_prob_calibrated': probs_cal,
    'fold': fold_ids
}).to_csv(os.path.join(out_dir, "SMOTE_OOF_predictions.csv"),
          index=False, encoding='utf-8-sig')

print("\n[已保存 SMOTE 主实验 4 个文件]")

# ================= DeLong 对比 SMOTE vs class_weight =================
def compute_midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x)
    T = np.zeros(N, dtype=float); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float); T2[J] = T
    return T2

def fastDeLong(pred_sorted, label_1_count):
    m = label_1_count; n = pred_sorted.shape[1] - m
    pos = pred_sorted[:, :m]; neg = pred_sorted[:, m:]
    k = pred_sorted.shape[0]
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m + n])
    for r in range(k):
        tx[r] = compute_midrank(pos[r])
        ty[r] = compute_midrank(neg[r])
        tz[r] = compute_midrank(pred_sorted[r])
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.cov(v01); sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov

def delong_roc_test(y_true, prob1, prob2):
    y_true = np.asarray(y_true).ravel()
    prob1 = np.asarray(prob1).ravel()
    prob2 = np.asarray(prob2).ravel()
    order = np.argsort(-y_true)
    label_1_count = int(y_true.sum())
    pred_sorted = np.vstack([prob1, prob2])[:, order]
    aucs, delongcov = fastDeLong(pred_sorted, label_1_count)
    l = np.array([[1, -1]])
    z = float(np.abs(np.diff(aucs))[0] /
              np.sqrt(np.dot(np.dot(l, delongcov), l.T)).ravel()[0])
    p = float(2 * stats.norm.sf(z))
    return aucs, z, p

print("\n" + "=" * 80)
print("DeLong：SMOTE vs class_weight='balanced'")
print("=" * 80)

# 加载 class_weight 版本的 Optuna OOF
cw = pd.read_csv(os.path.join(out_dir, "Optuna_calibrated_predictions.csv"))
y_cw = cw["y_true"].values.astype(int)
assert np.array_equal(y_cw, y), "OOF 索引不一致"
p_cw_raw = cw["y_prob_raw"].values.astype(float)
p_cw_cal = cw["y_prob_calibrated"].values.astype(float)

rows = []
def report(name, y_, p1, p2, l1, l2):
    aucs, z, p = delong_roc_test(y_, p1, p2)
    rows.append(dict(Comparison=name,
                     AUC_1=aucs[0], AUC_2=aucs[1],
                     Delta_AUC=aucs[0]-aucs[1], Z=z, p_value=p,
                     Significant=("Yes" if p < 0.05 else "No")))
    print(f"[{name}]  {l1}={aucs[0]:.4f}  {l2}={aucs[1]:.4f}  "
          f"Δ={aucs[0]-aucs[1]:+.4f}  Z={z:.3f}  p={p:.4g}")

report("Raw: SMOTE vs class_weight",
       y, probs_raw, p_cw_raw, "SMOTE", "cw")
report("Calibrated: SMOTE vs class_weight",
       y, probs_cal, p_cw_cal, "SMOTE_cal", "cw_cal")

pd.DataFrame(rows).to_csv(os.path.join(out_dir, "DeLong_SMOTE_vs_classweight.csv"),
                          index=False, encoding='utf-8-sig')

# ================= Bootstrap 95% CI =================
print("\n" + "=" * 80)
print("Bootstrap 95% CI (1000 次) — SMOTE")
print("=" * 80)
def bootstrap_ci(y_, p_, fn, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_); scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y_[idx])) < 2: continue
        scores.append(fn(y_[idx], p_[idx]))
    scores = np.array(scores)
    return scores.mean(), np.percentile(scores, 2.5), np.percentile(scores, 97.5)

for label, p in [("SMOTE-Raw", probs_raw), ("SMOTE-Calibrated", probs_cal)]:
    print(f"\n[{label}]")
    for mname, fn in [("AUC", roc_auc_score),
                      ("PR-AUC", average_precision_score),
                      ("Brier", brier_score_loss)]:
        m, lo, hi = bootstrap_ci(y, p, fn)
        print(f"    {mname:8s} = {m:.4f}   95% CI [{lo:.4f}, {hi:.4f}]")

# ================= SMOTE 版本 SHAP =================
print("\n" + "=" * 80)
print("SMOTE 版本 SHAP 分析")
print("=" * 80)

# 复用上面的 5 折循环，但这次保存 SHAP
shap_all = np.zeros((len(y), len(all_numeric)))
gkf2 = GroupKFold(n_splits=5)
for fold, (tr, te) in enumerate(gkf2.split(X_all, y, groups), 1):
    X_tr_raw, X_te_raw = X_all[tr], X_all[te]
    y_tr, y_te = y[tr], y[te]

    imp = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(X_tr_raw)
    X_te = imp.transform(X_te_raw)

    smote = SMOTE(sampling_strategy=SMOTE_RATIO, k_neighbors=SMOTE_K,
                  random_state=42, n_jobs=-1)
    X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)

    m = lgb.LGBMClassifier(**best_params)
    m.fit(X_tr_res, y_tr_res)

    contrib = m.predict(X_te, pred_contrib=True)
    shap_all[te] = contrib[:, :-1]
    print(f"  Fold {fold} SHAP done")

mean_abs_shap = pd.Series(np.abs(shap_all).mean(axis=0), index=all_numeric).sort_values(ascending=False)
mean_signed_shap = pd.Series(shap_all.mean(axis=0), index=all_numeric)

importance_df = pd.DataFrame({
    "Feature": mean_abs_shap.index,
    "MeanAbsSHAP": mean_abs_shap.values,
    "MeanSignedSHAP": [mean_signed_shap[f] for f in mean_abs_shap.index],
    "IsMDA": [f in MDA_FEATURES for f in mean_abs_shap.index],
})
importance_df["Rank"] = range(1, len(importance_df) + 1)
importance_df.to_csv(os.path.join(out_dir, "SMOTE_SHAP_global_importance.csv"),
                     index=False, encoding='utf-8-sig')

print("\nSMOTE SHAP 全局 Top 15:")
print(importance_df.head(15).to_string(index=False))

mda_imp = importance_df[importance_df["IsMDA"]]
print("\nSMOTE MD&A 特征 SHAP 重要性:")
print(mda_imp.to_string(index=False))
mda_imp.to_csv(os.path.join(out_dir, "SMOTE_SHAP_MDA_importance.csv"),
               index=False, encoding='utf-8-sig')

# 保存 SHAP 值
shap_df = pd.DataFrame(shap_all, columns=all_numeric)
shap_df.insert(0, "y_true", y)
shap_df.insert(1, "y_prob", probs_raw)
shap_df.to_csv(os.path.join(out_dir, "SMOTE_SHAP_values_all.csv"),
               index=False, encoding='utf-8-sig')

print("\n全部完成。输出文件:")
print("  SMOTE_calibration_raw.csv")
print("  SMOTE_calibration_calibrated.csv")
print("  SMOTE_fold_info.csv")
print("  SMOTE_OOF_predictions.csv")
print("  DeLong_SMOTE_vs_classweight.csv")
print("  SMOTE_SHAP_global_importance.csv")
print("  SMOTE_SHAP_MDA_importance.csv")
print("  SMOTE_SHAP_values_all.csv")

特征数: 83   样本数: 48328   正例: 4233
MD&A 特征数: 26

===== Fold 1 =====
  SMOTE: 38662 -> 70636  (pos: 3344 -> 35318)
  thr=0.166
  raw  AUC=0.7703 PR=0.2904 Brier=0.0790
  cal  AUC=0.7694 PR=0.2787 F1=0.3247 R=0.4094 P=0.2690 Spec=0.8873 Brier=0.0747
  耗时: 0.22 min

===== Fold 2 =====
  SMOTE: 38662 -> 70436  (pos: 3444 -> 35218)
  thr=0.169
  raw  AUC=0.7674 PR=0.2444 Brier=0.0752
  cal  AUC=0.7663 PR=0.2342 F1=0.3000 R=0.4778 P=0.2187 Spec=0.8483 Brier=0.0688
  耗时: 0.44 min

===== Fold 3 =====
  SMOTE: 38662 -> 70578  (pos: 3373 -> 35289)
  thr=0.171
  raw  AUC=0.7799 PR=0.2928 Brier=0.0755
  cal  AUC=0.7800 PR=0.2832 F1=0.3371 R=0.3988 P=0.2919 Spec=0.9055 Brier=0.0722
  耗时: 0.66 min

===== Fold 4 =====
  SMOTE: 38663 -> 70544  (pos: 3391 -> 35272)
  thr=0.164
  raw  AUC=0.7609 PR=0.2715 Brier=0.0766
  cal  AUC=0.7598 PR=0.2580 F1=0.3301 R=0.4264 P=0.2693 Spec=0.8896 Brier=0.0719
  耗时: 0.94 min

===== Fold 5 =====
  SMOTE: 38663 -> 70566  (pos: 3380 -> 35283)
  thr=0.166
  raw  AUC=0.7709

In [2]:
# ============ threadpoolctl monkey-patch ============
import threadpoolctl
class _SafeThreadpoolLimits:
    def __init__(self, *args, **kwargs): pass
    def __enter__(self): return self
    def __exit__(self, *args): pass
threadpoolctl.threadpool_limits = _SafeThreadpoolLimits
# =====================================================

import os, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              precision_recall_curve, f1_score, recall_score,
                              precision_score, confusion_matrix, brier_score_loss)
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\建模数据集_方案A_MDA_未隔离.csv"
OUT  = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"

data = pd.read_csv(DATA)
exclude_cols = ["Stkcd","year","Fraud","ShortName","IndustryName1","ViolationTypeID",
                "DeclareDate","DisposalDate","Enddate","set"]
feats = [c for c in data.columns
         if c not in exclude_cols and pd.api.types.is_numeric_dtype(data[c])]

# ---- 时间切分 ----
train_df = data[data["year"].between(2015, 2021)].copy()
val_df   = data[data["year"] == 2022].copy()
test_df  = data[data["year"].between(2023, 2024)].copy()

print(f"Train 2015-2021: {len(train_df)}  (pos={int(train_df['Fraud'].sum())})")
print(f"Val   2022:      {len(val_df)}    (pos={int(val_df['Fraud'].sum())})")
print(f"Test  2023-2024: {len(test_df)}   (pos={int(test_df['Fraud'].sum())})")

X_tr = train_df[feats].values; y_tr = train_df["Fraud"].values; g_tr = train_df["Stkcd"].values
X_va = val_df[feats].values;   y_va = val_df["Fraud"].values
X_te = test_df[feats].values;  y_te = test_df["Fraud"].values

# ---- 中位数填充（用训练集拟合）----
imp = SimpleImputer(strategy='median')
X_tr = imp.fit_transform(X_tr); X_va = imp.transform(X_va); X_te = imp.transform(X_te)

# ============ Stage 1: Optuna（只碰训练集）============
def objective(trial):
    params = dict(
        n_estimators    = trial.suggest_int('n_estimators', 200, 600, step=100),
        learning_rate   = trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
        num_leaves      = trial.suggest_int('num_leaves', 15, 50),
        min_child_samples = trial.suggest_int('min_child_samples', 10, 40),
        subsample       = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree= trial.suggest_float('colsample_bytree', 0.6, 1.0),
        reg_alpha       = trial.suggest_float('reg_alpha', 0.0, 0.5),
        reg_lambda      = trial.suggest_float('reg_lambda', 0.0, 0.5),
        class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1,
    )
    inner = GroupKFold(n_splits=3)
    scores = []
    for i_tr, i_val in inner.split(X_tr, y_tr, g_tr):
        m = lgb.LGBMClassifier(**params).fit(X_tr[i_tr], y_tr[i_tr])
        p = m.predict_proba(X_tr[i_val])[:,1]
        scores.append(average_precision_score(y_tr[i_val], p))
    return float(np.mean(scores))

t0 = time.time()
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30, show_progress_bar=False)
best_params = {**study.best_params,
               'class_weight': 'balanced', 'random_state': 42,
               'n_jobs': -1, 'verbose': -1}
print(f"\nOptuna best PR-AUC (inner 3-fold) = {study.best_value:.4f}")
print(f"Optuna 耗时: {(time.time()-t0)/60:.2f} min")

# ============ Stage 2: 用最优参数在全部训练集重训 ============
m_final = lgb.LGBMClassifier(**best_params).fit(X_tr, y_tr)
p_val_raw = m_final.predict_proba(X_va)[:,1]
p_te_raw  = m_final.predict_proba(X_te)[:,1]

# ============ Stage 3: Isotonic 校准（在验证集上拟合）============
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(p_val_raw, y_va)
p_val_cal = iso.predict(p_val_raw)
p_te_cal  = iso.predict(p_te_raw)

# ============ Stage 4: 阈值（在验证集选，F1 最优）============
def best_thr(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2*prec[:-1]*rec[:-1]/(prec[:-1]+rec[:-1]+1e-9)
    return float(thr[np.argmax(f1)])

thr = best_thr(y_va, p_val_cal)
print(f"\n验证集 F1 最优阈值 = {thr:.4f}")

# ============ Stage 5: 测试集只评估一次 ============
def eval_block(y, p, thr, name):
    y_pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    r = dict(Set=name, N=len(y), Positive=int(y.sum()),
             AUC=roc_auc_score(y,p), PR_AUC=average_precision_score(y,p),
             Brier=brier_score_loss(y,p),
             F1=f1_score(y,y_pred,zero_division=0),
             Recall=recall_score(y,y_pred,zero_division=0),
             Precision=precision_score(y,y_pred,zero_division=0),
             Specificity=tn/(tn+fp) if (tn+fp)>0 else np.nan)
    return r

rows = [
    eval_block(y_tr, m_final.predict_proba(X_tr)[:,1], 0.5, "Train"),
    eval_block(y_va, p_val_raw, 0.5, "Val (raw)"),
    eval_block(y_va, p_val_cal, thr, "Val (calibrated)"),
    eval_block(y_te, p_te_raw, 0.5, "Test (raw)"),
    eval_block(y_te, p_te_cal, thr, "Test (calibrated)"),
]
df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUT, "TimeSplit_metrics.csv"),
          index=False, encoding='utf-8-sig')
print("\n" + "="*90)
print(df.round(4).to_string(index=False))

# ============ 保存 OOF 和参数 ============
pd.DataFrame({'y_true': y_va, 'prob_raw': p_val_raw, 'prob_cal': p_val_cal}
             ).to_csv(os.path.join(OUT, "TimeSplit_val_predictions.csv"),
                      index=False, encoding='utf-8-sig')
pd.DataFrame({'y_true': y_te, 'prob_raw': p_te_raw, 'prob_cal': p_te_cal}
             ).to_csv(os.path.join(OUT, "TimeSplit_test_predictions.csv"),
                      index=False, encoding='utf-8-sig')
pd.DataFrame([best_params]).to_csv(os.path.join(OUT, "TimeSplit_best_params.csv"),
                                    index=False, encoding='utf-8-sig')
print("\n已保存: TimeSplit_metrics.csv / TimeSplit_val_predictions.csv /")
print("        TimeSplit_test_predictions.csv / TimeSplit_best_params.csv")

Train 2015-2021: 31553  (pos=2789)
Val   2022:      5538    (pos=528)
Test  2023-2024: 11237   (pos=916)

Optuna best PR-AUC (inner 3-fold) = 0.3221
Optuna 耗时: 2.82 min

验证集 F1 最优阈值 = 0.2518

              Set     N  Positive    AUC  PR_AUC  Brier     F1  Recall  Precision  Specificity
            Train 31553      2789 0.9263  0.5605 0.1277 0.4766  0.9057     0.3234       0.8163
        Val (raw)  5538       528 0.7855  0.2985 0.1882 0.3261  0.7121     0.2115       0.7202
 Val (calibrated)  5538       528 0.7917  0.2943 0.0753 0.3751  0.4564     0.3184       0.8970
       Test (raw) 11237       916 0.7578  0.2506 0.2112 0.2622  0.7183     0.1603       0.6661
Test (calibrated) 11237       916 0.7563  0.2337 0.0693 0.3135  0.4520     0.2400       0.8730

已保存: TimeSplit_metrics.csv / TimeSplit_val_predictions.csv /
        TimeSplit_test_predictions.csv / TimeSplit_best_params.csv
